[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto2/2pipeline.ipynb)
Confirmed runtime version: 2026.04

## Setup

In [ ]:
import sys

vi = sys.version_info
if not ((3, 12) <= (vi.major, vi.minor) < (3, 13)):
    raise RuntimeError(f"Python 3.12 required, got {vi.major}.{vi.minor}.{vi.micro}")

print(f"Python {vi.major}.{vi.minor}.{vi.micro} ✓")

In [ ]:
!pip install -q transformers torch spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

print("Setup complete.")

## Data Models

In [ ]:
class Role(str, Enum):
    TECHNICAL_METHOD = "technical_method"
    TASK = "task"
    DATASET = "dataset"
    EVALUATION_METRIC = "evaluation_metric"
    OTHER = "other"


@dataclass
class CandidateWithContext:
    candidate: str
    sentence: str
    section: str = "unknown"


@dataclass
class MethodologyProfile:
    technical_method: list[str] = field(default_factory=list)
    task: list[str] = field(default_factory=list)
    dataset: list[str] = field(default_factory=list)
    evaluation_metric: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "TechnicalMethod": self.technical_method,
            "Task": self.task,
            "Dataset": self.dataset,
            "EvaluationMetric": self.evaluation_metric,
        }


print("Models ready.")

## Model Setup

In [ ]:
import spacy
from transformers import pipeline

nlp = spacy.load("en_core_web_sm")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
)
print("Models ready.")

## Step 0 — Load TEI XML

Upload a TEI XML file produced by local GROBID.

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}
SKIP_KEYWORDS = {"related work", "related works"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []
skip_prefix = None

for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    h_el = div.find("tei:head", NS)
    n_attr = (h_el.get("n", "") if h_el is not None else "").rstrip(".")
    h_lower = heading.lower().strip()

    if h_lower in SKIP_HEADINGS:
        skip_prefix = None
        continue

    if any(kw in h_lower for kw in SKIP_KEYWORDS):
        skip_prefix = n_attr if n_attr else None
        continue

    if skip_prefix:
        if n_attr and n_attr.startswith(skip_prefix + "."):
            continue
        elif n_attr:
            skip_prefix = None

    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded : {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Step 0b — Select Sections (all body sections)

In [ ]:
target_sections = sections

print(f"Target sections: {len(target_sections)}")
for s in target_sections:
    print(f"  - {s['heading']}")

## Step 0c — Sentence Splitting with spaCy

In [ ]:
import re


def pre_clean(text: str) -> str:
    return re.sub(r"\s*\[\d+(?:[,\s]*\d+)*\]\s*", " ", text).strip()


def is_valid(text: str, min_len: int = 30) -> bool:
    if len(text) < min_len:
        return False
    if text.startswith(("†", "‡")):
        return False
    return bool(re.search(r"[a-zA-Z]{3,}", text))


sentences: list[CandidateWithContext] = []

for s in target_sections:
    doc = nlp(pre_clean(s["text"]))
    for sent in doc.sents:
        text = sent.text.strip()
        if is_valid(text):
            sentences.append(
                CandidateWithContext(candidate="", sentence=text, section=s["heading"])
            )

print(f"Total sentences: {len(sentences)}")
for c in sentences[:5]:
    print(f"  [{c.section}] {c.sentence[:80]}")

## Step 2 — Classify Sentences

In [ ]:
LABELS_SHORT = ["technical method", "dataset", "evaluation metric", "task"]

_TM = "This sentence describes a technique, algorithm, system, or architecture used or proposed in the research."  # noqa: E501
_DS = "This sentence describes data, a dataset, or corpus used in the research."
_EM = "This sentence describes a metric, measure, or criterion used to evaluate results or performance."  # noqa: E501
_TA = "This sentence describes the problem, task, or objective the research addresses."

LABELS_VERBOSE = [_TM, _DS, _EM, _TA]

VERBOSE_TO_ROLE: dict[str, Role] = {
    _TM: Role.TECHNICAL_METHOD,
    _DS: Role.DATASET,
    _EM: Role.EVALUATION_METRIC,
    _TA: Role.TASK,
}

SHORT_TO_ROLE: dict[str, Role] = {
    "technical method": Role.TECHNICAL_METHOD,
    "dataset": Role.DATASET,
    "evaluation metric": Role.EVALUATION_METRIC,
    "task": Role.TASK,
}

THRESHOLD = 0.5

profile = MethodologyProfile()
results_verbose = []

for c in sentences:
    result = classifier(
        c.sentence,
        candidate_labels=LABELS_VERBOSE,
        hypothesis_template="{}",
    )
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    role = VERBOSE_TO_ROLE[top_label]
    accepted = top_score >= THRESHOLD
    results_verbose.append((c, role, top_score, accepted))
    mark = "✓" if accepted else "✗"
    snippet = c.sentence[:55].ljust(55)
    sec = c.section[:14].ljust(14)
    print(f"[{sec}] {snippet}  {role.value:20}  {top_score:.2f} {mark}")
    if accepted:
        if role == Role.TECHNICAL_METHOD:
            profile.technical_method.append(c.sentence)
        elif role == Role.TASK:
            profile.task.append(c.sentence)
        elif role == Role.DATASET:
            profile.dataset.append(c.sentence)
        elif role == Role.EVALUATION_METRIC:
            profile.evaluation_metric.append(c.sentence)

print()
print(profile.to_dict())

## Step 2b — Comparison: Short vs Verbose Hypotheses

In [ ]:
# === Comparison: short labels vs verbose hypotheses ===
diffs = []
for c, role_verbose, score_verbose, _ in results_verbose:
    r = classifier(c.sentence, candidate_labels=LABELS_SHORT)
    role_short = SHORT_TO_ROLE[r["labels"][0]]
    if role_short != role_verbose:
        diffs.append((c.sentence, role_short, role_verbose))

print(f"Changed: {len(diffs)} / {len(results_verbose)} sentences")
print()
for sentence, r_short, r_verbose in diffs:
    print(f"  short  → {r_short.value}")
    print(f"  verbose→ {r_verbose.value}")
    print(f"  {sentence[:120]}")
    print()